# Macro CPI Data Pipeline & Quality Diagnostics

Builds monthly inflation series (`cpi_mom`, `cpi_yoy`, `cpi_index`) from official Bank of Russia statistical releases via `ofz_curve.cpi_loaders` with publication-lag alignment.

In [ ]:
import warnings
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from ofz_curve.cpi_loaders import get_cbr_inflation_data
from ofz_curve.paths import RUSSIA_CPI_CSV

warnings.filterwarnings('ignore')

In [ ]:
cpi_data = get_cbr_inflation_data()
if cpi_data is None:
    raise RuntimeError('Не удалось собрать CPI из файлов ЦБ РФ')

print(f'Период: {cpi_data.index.min().date()} .. {cpi_data.index.max().date()}')
print(f'Наблюдений: {len(cpi_data)}')
display(cpi_data.head())
display(cpi_data.tail())

In [ ]:
# Контроль качества данных
is_unique_index = not cpi_data.index.duplicated().any()
has_2020 = (cpi_data.index.year == 2020).any()
mom_equals_yoy_all = (cpi_data['cpi_mom'] == cpi_data['cpi_yoy']).all()

print('Уникальный monthly index:', is_unique_index)
print('Присутствует 2020 год:', has_2020)
print('Полное совпадение cpi_mom и cpi_yoy:', mom_equals_yoy_all)

if not is_unique_index:
    raise AssertionError('В CPI найдены дубли дат')
if not has_2020:
    raise AssertionError('В CPI отсутствуют наблюдения 2020 года')
if mom_equals_yoy_all:
    raise AssertionError('cpi_mom и cpi_yoy полностью совпадают - данные некорректны')

display(cpi_data.describe())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
cpi_data['cpi_mom'].plot(ax=axes[0, 0], color='#1f77b4', title='Инфляция м/м (cpi_mom)')
cpi_data['cpi_yoy'].plot(ax=axes[0, 1], color='#d62728', title='Инфляция г/г (cpi_yoy)')
cpi_data['cpi_index'].plot(ax=axes[1, 0], color='#2ca02c', title='Индекс CPI (база=100)')
(cpi_data['cpi_mom'] - cpi_data['cpi_yoy']).plot(ax=axes[1, 1], color='#9467bd', title='Разность cpi_mom - cpi_yoy')
for ax in axes.flat:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Дополнительный анализ по периодам (полезно для интерпретации в тексте)
periods_analysis = [
    ('2013-2014', '2013-01-01', '2014-12-31', 'предкризисный/инфляционное ускорение'),
    ('2015', '2015-01-01', '2015-12-31', 'шоковый год'),
    ('2016-2019', '2016-01-01', '2019-12-31', 'дезинфляция и стабилизация'),
    ('2020-2021', '2020-01-01', '2021-12-31', 'пандемийный период'),
    ('2022-2025', '2022-01-01', '2025-12-31', 'новый инфляционный режим'),
]

print('Статистика по периодам:')
for name, start, end, desc in periods_analysis:
    sub = cpi_data[(cpi_data.index >= start) & (cpi_data.index <= end)]
    if len(sub) == 0:
        continue
    print(f'\n{name} ({desc}), n={len(sub)}')
    print(f'  cpi_mom mean: {sub["cpi_mom"].mean()*100:.2f}%')
    print(f'  cpi_yoy mean: {sub["cpi_yoy"].mean()*100:.2f}%')
    print(f'  cpi_mom std : {sub["cpi_mom"].std()*100:.2f}%')

In [ ]:
cpi_data.to_csv(RUSSIA_CPI_CSV)
print(f'Файл сохранен: {RUSSIA_CPI_CSV}')
print('Пропуски по колонкам:')
display(cpi_data.isna().sum())